In [ ]:
# Group 5 VIA cervical screening - shared team notebook
# 1. Runtime > Change runtime type > T4 GPU
# 2. Run the three cells below in order (sign in to Google Drive when asked)
# If TEAM isn't found: in Google Drive, open the 'AI ML Final Project' folder menu > Organise > Add shortcut > My Drive
from google.colab import drive
import os, torch

drive.mount('/content/drive')

TEAM = '/content/drive/MyDrive/AI ML Final Project'
FINAL = f'{TEAM}/Finalized Dataset'  # every dataset we use: IARC VIA, Jhpiego VIA, IARC colposcopy, AnnoCerv
DATA = '/content/data'

assert os.path.isdir(TEAM), 'Team folder not in My Drive yet - add the shortcut (see comment above)'
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none - switch the runtime to T4')
print('Datasets:', sorted(os.listdir(FINAL)))

In [ ]:
# Copy the two labelled VIA datasets to the runtime's local disk; reading from Drive during training is much slower
# The loader expects the Jhpiego flashcards in a folder called 'via'
import shutil

for src, dst in [('IARCImageBankVIA', 'IARCImageBankVIA'), ('JHPIEGOVIA', 'via')]:
    if not os.path.exists(f'{DATA}/{dst}'):
        shutil.copytree(f'{FINAL}/{src}', f'{DATA}/{dst}')
for name in sorted(os.listdir(DATA)):
    print(name, sum(len(f) for _, _, f in os.walk(f'{DATA}/{name}')), 'files')

In [ ]:
# IARC VIA Image Bank: one row per case in meta, two photos per case (before/after acetic acid) in images
import pandas as pd

meta = pd.read_excel(f'{DATA}/IARCImageBankVIA/Cases Meta data.xlsx')
images = pd.read_excel(f'{DATA}/IARCImageBankVIA/Cases - Images.xlsx')
print('meta', meta.shape, '| images', images.shape)
print(list(meta.columns))
meta.head()

## Work area
Add your work below under a heading with your name and task (e.g. "## Kavin - classification baseline").

- Colab doesn't merge edits: avoid editing the same cell as someone else at the same time.
- For long experiments, use **File > Save a copy in Drive**, then move the copy into the *AI ML Final Project* folder.
- Save model checkpoints to `TEAM` (Drive); everything in `/content` is wiped when the runtime stops.

## Kavin - EfficientNet-B3 baseline + pruning (Nov 11)
Plan from the Nov 4 meeting: train EfficientNet-B3, then prune it and see how small it can get before accuracy drops.

- Same 300 images (IARC + Jhpiego), same 5 folds, training recipe and metrics as the preliminary notebook, so the numbers compare directly with B0 and MobileNetV3.
- Pruning: 3 rounds removing 20% of channels each (20%, 40%, 60% total), with a 6-epoch fine-tune after each round. Whole channels are deleted, so the model really gets smaller.
- Target: sensitivity of at least 73% at 86.7% specificity. Best so far: MobileNetV3-Large (AUC 0.856, sensitivity 0.64, 3-class accuracy 0.730, 4.2M parameters).
- Run the three setup cells at the top first. Results are saved to AI ML Final Project/results/b3_pruning. About 30 minutes on a T4.

In [ ]:
# Data: same 300 labelled images as the preliminary notebook (loader and labels come from our GitHub repo)
!pip -q install torch-pruning
import time, random, urllib.request
import numpy as np, pandas as pd, torch, torch.nn as nn
import torch_pruning as tp
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms as T
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score

REPO = 'https://raw.githubusercontent.com/kavinraj1/group5-via-cervical-screening/main'
for f in ['src/via_data.py', 'labels/jhpiego_labels.csv']:
    urllib.request.urlretrieve(f'{REPO}/{f}', f.split('/')[-1])
from via_data import load_table, open_image, CLASSES

def shrink(img, short=400):
    w, h = img.size
    s = short / min(w, h)
    return img.resize((round(w * s), round(h * s)))

df = load_table(DATA, 'jhpiego_labels.csv')
images = [shrink(open_image(r)) for _, r in df.iterrows()]
labels, refer = df['label'].to_numpy(), df['refer'].to_numpy()

SEED, N_FOLDS = 42, 5
strata = (df['source'] + '_' + df['label'].astype(str)).to_numpy()
folds = np.zeros(len(df), dtype=int)
for k, (_, te) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(df, strata)):
    folds[te] = k
print(len(df), 'images |', dict(zip(CLASSES, np.bincount(labels))))

In [ ]:
# Training recipe from the preliminary notebook, with B3 at its native 300 px input
device = torch.device('cuda')
IMG, BATCH, EPOCHS, LR = 300, 16, 20, 1e-3
MIN_SPEC, MIN_SENS = 0.867, 0.73
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = T.Compose([
    T.RandomResizedCrop(IMG, scale=(0.6, 1.0), ratio=(0.8, 1.25)),
    T.RandomHorizontalFlip(), T.RandomRotation(15),
    T.ColorJitter(0.2, 0.2, 0.1, 0.02),  # small hue change: acetowhite colour is the signal
    T.ToTensor(), T.Normalize(MEAN, STD)])
eval_tf = T.Compose([T.Resize(IMG), T.CenterCrop(IMG), T.ToTensor(), T.Normalize(MEAN, STD)])

class ImgDS(Dataset):
    def __init__(self, idx, tf): self.idx, self.tf = np.asarray(idx), tf
    def __len__(self): return len(self.idx)
    def __getitem__(self, i): return self.tf(images[self.idx[i]]), labels[self.idx[i]]

def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def make_b3():
    m = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, 3)
    return m.to(device)

def train(model, idx, epochs, lr, l1=0.0):
    # l1 > 0: L1 penalty on BatchNorm scales pushes channels the task doesn't use toward zero
    bns = [m for m in model.modules() if isinstance(m, nn.BatchNorm2d)]
    head = [p for n, p in model.named_parameters() if n.startswith('classifier')]
    body = [p for n, p in model.named_parameters() if not n.startswith('classifier')]
    opt = torch.optim.AdamW([{'params': body, 'lr': lr * 0.3}, {'params': head, 'lr': lr}], weight_decay=1e-4)
    dl = DataLoader(ImgDS(idx, train_tf), batch_size=BATCH, shuffle=True, num_workers=2, drop_last=True)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[lr * 0.3, lr], total_steps=epochs * len(dl), pct_start=0.15)
    counts = np.bincount(labels[idx], minlength=3)
    weights = torch.tensor(len(idx) / (3 * counts), dtype=torch.float32, device=device)
    loss_fn = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)
    scaler = torch.amp.GradScaler('cuda')
    for _ in range(epochs):
        model.train()
        for x, y in dl:
            x, y = x.to(device), torch.as_tensor(y).to(device)
            with torch.autocast('cuda', dtype=torch.float16):
                loss = loss_fn(model(x), y)
                if l1:
                    loss = loss + l1 * sum(m.weight.abs().sum() for m in bns)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
    return model

@torch.no_grad()
def predict(model, idx):
    model.eval()
    out = []
    for x, _ in DataLoader(ImgDS(idx, eval_tf), batch_size=32, num_workers=2):
        x = x.to(device)
        with torch.autocast('cuda', dtype=torch.float16):
            p = model(x).float().softmax(1) + model(torch.flip(x, dims=[3])).float().softmax(1)
        out.append((p / 2).cpu().numpy())
    return np.concatenate(out)

def sens_at_spec(y, s, spec):
    fpr, tpr, _ = roc_curve(y, s)
    return tpr[fpr <= 1 - spec + 1e-9].max()

In [ ]:
# 5-fold CV: train B3, then delete 20% of channels per round (3 rounds), fine-tuning after each round
PRUNE_STEPS, FT_EPOCHS = 3, 6
levels = ['B3 full'] + [f'B3 pruned {20 * (s + 1)}%' for s in range(PRUNE_STEPS)]
probs = {name: np.zeros((len(df), 3)) for name in levels}
size = {}
example = torch.randn(1, 3, IMG, IMG, device=device)

seed_all()
t0 = time.time()
for k in range(N_FOLDS):
    tr, te = np.where(folds != k)[0], np.where(folds == k)[0]
    model = train(make_b3(), tr, EPOCHS, LR)
    probs['B3 full'][te] = predict(model, te)
    size['B3 full'] = tp.utils.count_ops_and_params(model, example)
    pruner = tp.pruner.MetaPruner(
        model, example, importance=tp.importance.GroupMagnitudeImportance(p=2),
        pruning_ratio=0.2 * PRUNE_STEPS, iterative_steps=PRUNE_STEPS,
        ignored_layers=[model.classifier], round_to=8)
    for s in range(PRUNE_STEPS):
        pruner.step()
        model = train(model, tr, FT_EPOCHS, LR / 3)
        probs[levels[s + 1]][te] = predict(model, te)
        size[levels[s + 1]] = tp.utils.count_ops_and_params(model, example)
    print(f'fold {k + 1}/{N_FOLDS} done ({(time.time() - t0) / 60:.1f} min)')

In [ ]:
# Accuracy vs size at each pruning level (out-of-fold predictions on all 300 images)
import pandas as pd
rows = []
for name in levels:
    p = probs[name]
    s = p[:, 1] + p[:, 2]
    macs, params = size[name]
    rows.append({'model': name,
                 'params (M)': round(params / 1e6, 2),
                 'GMACs': round(macs / 1e9, 2),
                 'int8 size (MB, est.)': round(params / 1e6, 1),
                 'AUC': round(roc_auc_score(refer, s), 3),
                 'sens @ 86.7% spec': round(sens_at_spec(refer, s, MIN_SPEC), 2),
                 '3-class acc': round(accuracy_score(labels, p.argmax(1)), 3)})
res = pd.DataFrame(rows).set_index('model')
res['meets target'] = res['sens @ 86.7% spec'] >= MIN_SENS
os.makedirs(f'{TEAM}/results/b3_pruning', exist_ok=True)
res.to_csv(f'{TEAM}/results/b3_pruning/results.csv')
res

## Kavin - targeted pruning of B3 (Nov 11)
The uniform run above removed 20% of channels from every layer, ranked by weight size (mostly ImageNet knowledge), and lost a lot of accuracy. This version tries to remove only the channels B3 doesn't need for VIA:

- **Sparsity training:** an L1 penalty on BatchNorm scales during training pushes channels the task doesn't use toward zero.
- **Taylor importance:** each channel is scored by how much the loss on our VIA training images would rise if it were removed.
- **Global pruning:** channels are ranked across the whole network, so redundant layers lose more and important layers keep theirs.
- **Stop on accuracy, not a fixed %:** remove 5% of channels per step (3-epoch fine-tune after each), and keep the deepest step whose validation AUC stays within 0.03 of the unpruned model. The chosen model gets a final 6-epoch fine-tune on the full training fold.
- Validation images come from each fold's training images (20%), never the test fold. With ~48 validation images the stopping point is noisy.
- Needs cells 0, 1, 5 and 6 run first. About 25 minutes on a T4.

In [ ]:
# Targeted pruning: sparsity-train B3, then remove the least important channels network-wide until validation AUC drops
import copy
from sklearn.model_selection import train_test_split

L1, STEP_PCT, MAX_STEPS, FT_EPOCHS, FINAL_EPOCHS, TOL = 1e-4, 5, 10, 3, 6, 0.03
example = torch.randn(1, 3, IMG, IMG, device=device)

def taylor_grads(model, idx):
    # Taylor importance reads |weight x gradient|, so accumulate loss gradients on VIA training images first
    model.eval()
    model.zero_grad(set_to_none=True)
    loss_fn = nn.CrossEntropyLoss()
    for x, y in DataLoader(ImgDS(idx, eval_tf), batch_size=BATCH, num_workers=2):
        loss_fn(model(x.to(device)), torch.as_tensor(y).to(device)).backward()

def val_auc(model, idx):
    p = predict(model, idx)
    return roc_auc_score(refer[idx], p[:, 1] + p[:, 2])

steps = [f'{STEP_PCT * s}% channels removed' for s in range(MAX_STEPS + 1)]
tprobs = {name: np.zeros((len(df), 3)) for name in steps + ['B3 targeted']}
tsize = {name: [] for name in tprobs}
chosen = []

seed_all()
t0 = time.time()
for k in range(N_FOLDS):
    tr, te = np.where(folds != k)[0], np.where(folds == k)[0]
    fit, val = train_test_split(tr, test_size=0.2, stratify=strata[tr], random_state=SEED)
    model = train(make_b3(), fit, EPOCHS, LR, l1=L1)
    pruner = tp.pruner.MetaPruner(
        model, example, importance=tp.importance.GroupTaylorImportance(),
        global_pruning=True, pruning_ratio=STEP_PCT * MAX_STEPS / 100, iterative_steps=MAX_STEPS,
        max_pruning_ratio=0.8, ignored_layers=[model.classifier], round_to=8)
    base = val_auc(model, val)
    best, best_step, stopped, trace = copy.deepcopy(model), 0, False, [round(base, 3)]
    for s in range(MAX_STEPS + 1):
        if s:
            taylor_grads(model, fit)
            pruner.step()
            model.zero_grad(set_to_none=True)
            model = train(model, fit, FT_EPOCHS, LR / 3)
            auc = val_auc(model, val)
            trace.append(round(auc, 3))
            if not stopped and auc >= base - TOL:
                best, best_step = copy.deepcopy(model), s
            else:
                stopped = True
        tprobs[steps[s]][te] = predict(model, te)
        tsize[steps[s]].append(tp.utils.count_ops_and_params(model, example))
    best = train(best, tr, FINAL_EPOCHS, LR / 3)
    tprobs['B3 targeted'][te] = predict(best, te)
    tsize['B3 targeted'].append(tp.utils.count_ops_and_params(best, example))
    chosen.append(STEP_PCT * best_step)
    print(f'fold {k + 1}/{N_FOLDS}: kept {STEP_PCT * best_step}% removed | val AUC per step {trace} ({(time.time() - t0) / 60:.1f} min)')

In [ ]:
# Accuracy vs size: each pruning step (curve) and the validation-chosen model per fold (B3 targeted)
import pandas as pd

def summarize(probs, size):
    rows = []
    for name in probs:
        p = probs[name]
        s = p[:, 1] + p[:, 2]
        macs, params = np.mean(size[name], axis=0)  # sizes differ by fold with global pruning
        rows.append({'model': name,
                     'params (M)': round(params / 1e6, 2),
                     'GMACs': round(macs / 1e9, 2),
                     'int8 size (MB, est.)': round(params / 1e6, 1),
                     'AUC': round(roc_auc_score(refer, s), 3),
                     'sens @ 86.7% spec': round(sens_at_spec(refer, s, MIN_SPEC), 2),
                     '3-class acc': round(accuracy_score(labels, p.argmax(1)), 3)})
    out = pd.DataFrame(rows).set_index('model')
    out['meets target'] = out['sens @ 86.7% spec'] >= MIN_SENS
    return out

tres = summarize(tprobs, tsize)
print('Channels removed in the chosen model, per fold:', chosen)
os.makedirs(f'{TEAM}/results/b3_pruning', exist_ok=True)
tres.to_csv(f'{TEAM}/results/b3_pruning/targeted_results.csv')
tres

In [ ]:
# Full metric set for the weekly brief: refer vs no-refer at the 86.7% specificity operating point, plus 3-class macro scores
# The threshold is picked on the same out-of-fold predictions it's scored on, so these are slightly optimistic
from sklearn.metrics import precision_recall_fscore_support

def op_threshold(y, s, spec):
    fpr, tpr, thr = roc_curve(y, s)
    ok = fpr <= 1 - spec + 1e-9
    return thr[ok][tpr[ok].argmax()]

def all_metrics(p):
    s = p[:, 1] + p[:, 2]
    pred, y = s >= op_threshold(refer, s, MIN_SPEC), refer == 1
    tp_, fp_, fn_, tn_ = (pred & y).sum(), (pred & ~y).sum(), (~pred & y).sum(), (~pred & ~y).sum()
    prec, rec = tp_ / max(tp_ + fp_, 1), tp_ / (tp_ + fn_)
    p3, r3, f3, _ = precision_recall_fscore_support(labels, p.argmax(1), average='macro', zero_division=0)
    return {'AUC': roc_auc_score(refer, s),
            'refer recall (sens)': rec, 'refer specificity': tn_ / (tn_ + fp_),
            'refer precision': prec, 'refer F1': 2 * prec * rec / max(prec + rec, 1e-9),
            'refer accuracy': (tp_ + tn_) / len(y),
            '3-class acc': accuracy_score(labels, p.argmax(1)),
            '3-class macro precision': p3, '3-class macro recall': r3, '3-class macro F1': f3}

models_all = {**probs, **{f'Targeted: {k}': v for k, v in tprobs.items()}}
metrics = pd.DataFrame({name: all_metrics(p) for name, p in models_all.items()}).T.round(3)
metrics.index.name = 'model'
metrics.to_csv(f'{TEAM}/results/b3_pruning/all_metrics.csv')
metrics

## Kavin - distillation using all datasets (Nov 11)
Pruning B3 lost accuracy at every step, so this tries the other route from the Nov 4 plan: train the full EfficientNet-B3 as a **teacher**, then train a phone-sized **MobileNetV3-Large student** (~4M params, ~4 MB at int8) to copy its predictions.

### Datasets (preprocessing)
| Dataset | Image type | Images available | Labelled (train + test) | Unlabelled, used for distillation |
|---|---|---|---|---|
| IARC VIA Image Bank | VIA | 417 | 185 | 5 |
| Jhpiego VIA flashcards | VIA | 243 | 115 | 0 |
| IARC Colposcopy Image Bank | Colposcopy | 913 | 0 | 323 |
| AnnoCerv | Colposcopy | 532 | 0 | 297 |
| **Total** | | **2,105** | **300** | **625** |

- **Labelled (300):** expert VIA readings (Negative / Positive / Suspicious of cancer). Same 300 images and 5 folds as every result above: ~240 train / ~60 test per fold.
- **Unlabelled (625):** every acetic-acid photo that isn't already in the labelled set. They have no VIA label, so the student only learns the teacher's soft predictions on them.
- **Training images per fold for the distilled student: ~865** (240 labelled + up to 625 teacher-labelled).
- **Not used:** before-acid, Lugol's iodine, green-filter and vulva photos (VIA is read after acetic acid), the Jhpiego answer-key pages (same photos as the labelled set) and info pages. AnnoCerv's lesion masks and Swede scores are kept for the segmentation head.
- AnnoCerv is cloned from GitHub because the Drive copy was still uploading.

### Method
- Teacher: EfficientNet-B3, same recipe as above, retrained inside each fold on that fold's training images only.
- Student loss: 0.3 x cross-entropy on labelled images + 0.7 x KL divergence to the teacher's softened predictions (temperature 2) on labelled and unlabelled images. Teacher and student see the same augmented view each step.
- No leakage: other photos of a test-fold patient are removed from that fold's unlabelled set, and all scores are on held-out test folds.
- BatchNorm statistics are recomputed on VIA training images after training, because the colposcopy photos look different.
- Compared: student with no distillation, distillation on labelled VIA only, and distillation with all datasets (does the extra data help?).
- Results are saved to AI ML Final Project/results/distillation. About 30 minutes on a T4.

### Results (5-fold CV, 300 labelled VIA images)
| Model | Params | AUC | Sens @ 86.7% spec | Refer F1 | 3-class acc | 3-class macro F1 |
|---|---|---|---|---|---|---|
| Teacher: EfficientNet-B3 | 10.7M | 0.885 | 0.73 | 0.787 | 0.763 | 0.769 |
| Student: MobileNetV3-L, no distillation | 4.2M | 0.851 | 0.64 | 0.724 | 0.723 | 0.706 |
| **Student + distillation (labelled VIA only)** | **4.2M** | **0.858** | **0.68** | **0.756** | **0.730** | **0.729** |
| Student + distillation (all datasets) | 4.2M | 0.848 | 0.61 | 0.700 | 0.687 | 0.666 |

- Distillation helps: the student gains ~5 points of sensitivity and ~2 points of macro F1 over the same network trained alone, at 40% of the teacher's size and ~4.5x less compute.
- Adding the 620 colposcopy photos made the student worse. The teacher was only trained on VIA, so its predictions on magnified colposcopy photos are noisy targets. Those datasets are better used for the segmentation head (AnnoCerv masks) or supervised pretraining.
- With 300 images, differences of ~0.01 AUC or a few points of sensitivity are within run-to-run noise (the same B3 recipe gave 0.69 sensitivity in the pruning run and 0.73 here).

In [ ]:
# Preprocessing: inventory of every dataset we have, and which images each part of the distillation uses
import subprocess
from concurrent.futures import ThreadPoolExecutor
from PIL import Image

FINAL = f'{TEAM}/Finalized Dataset'
ANNO = '/content/AnnoCerv/dataset'
if not os.path.isdir(ANNO):  # the Drive copy is still uploading, so clone the public repo (CC BY 4.0)
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/iclx/AnnoCerv.git', '/content/AnnoCerv'], check=True)

def is_acetic(t):
    # VIA is read after acetic acid, so only those photos are used (no Lugol's iodine, green filter, before-acid or vulva photos)
    t = str(t).strip().lower()
    return t.startswith('after') and 'acetic' in t and 'green' not in t

def file_index(root):
    return {f: os.path.join(d, f) for d, _, fs in os.walk(root) for f in fs}

def n_images(root):
    return sum(f.lower().endswith(('.jpg', '.jpeg', '.png')) for _, _, fs in os.walk(root) for f in fs)

def bank_extras(bank, prefix, skip):
    x = pd.read_excel(f'{bank}/Cases - Images.xlsx').rename(columns={'Case Number': 'CaseNumber'})
    idx = file_index(bank)
    return [(idx[r.File], f'{prefix}-{int(r.CaseNumber):03d}') for r in x.itertuples()
            if is_acetic(r.Type) and r.File in idx and idx[r.File] not in skip]

via_extra = bank_extras(f'{DATA}/IARCImageBankVIA', 'IARC', set(df['path']))
colpo_extra = bank_extras(f'{FINAL}/IARCImageBankColpo', 'COLPO', set())
anno_all = [p for p in file_index(ANNO).values() if p.lower().endswith('.jpg')]
anno_extra = [(p, 'ANNO-' + os.path.basename(os.path.dirname(p))) for p in anno_all if 'Aceto' in os.path.basename(p)]
extra = via_extra + colpo_extra + anno_extra

n_lab = df['source'].value_counts()
inv = pd.DataFrame([
    ['IARC VIA Image Bank', 'VIA', n_images(f'{DATA}/IARCImageBankVIA'), n_lab.get('IARC', 0), len(via_extra),
     "Expert VIA labels; before-acid and Lugol's iodine photos not used"],
    ['Jhpiego VIA flashcards', 'VIA', n_images(f'{DATA}/via'), n_lab.get('Jhpiego', 0), 0,
     'Expert VIA labels; answer-key pages (same photos) and info pages not used'],
    ['IARC Colposcopy Image Bank', 'Colposcopy', n_images(f'{FINAL}/IARCImageBankColpo'), 0, len(colpo_extra),
     'No VIA labels; acetic-acid photos get teacher soft labels'],
    ['AnnoCerv', 'Colposcopy', len(anno_all), 0, len(anno_extra),
     'Acetic-acid photos get teacher soft labels; lesion masks and Swede scores kept for the segmentation head'],
], columns=['dataset', 'image type', 'images available', 'labelled (train + test)', 'unlabelled (distillation only)', 'how it is used'])
inv.loc['total'] = ['All datasets', '', inv['images available'].sum(), inv['labelled (train + test)'].sum(),
                    inv['unlabelled (distillation only)'].sum(), '']

def load_small(p):
    return shrink(Image.open(p).convert('RGB'))

with ThreadPoolExecutor(16) as ex:
    extra_imgs = list(ex.map(load_small, [p for p, _ in extra]))
extra_case = np.array([c for _, c in extra])
n_train = len(df) * (N_FOLDS - 1) // N_FOLDS
print(f'Labelled VIA images: {len(df)} (5-fold CV: ~{n_train} train / ~{len(df) - n_train} test per fold)')
print(f'Unlabelled acetic-acid images for distillation: {len(extra)}')
print(f'Images the distilled student trains on per fold: ~{n_train + len(extra)} ({n_train} labelled + up to {len(extra)} teacher-labelled)')
inv

In [ ]:
# Distillation: EfficientNet-B3 teacher -> MobileNetV3-Large student, on the same 5 folds as every other result
import torch.nn.functional as F

KD_T, KD_ALPHA, S_EPOCHS, X_BATCH = 2.0, 0.3, 30, 32
example = torch.randn(1, 3, IMG, IMG, device=device)

class ExtraDS(Dataset):
    def __init__(self, idx): self.idx = np.asarray(idx)
    def __len__(self): return len(self.idx)
    def __getitem__(self, i): return train_tf(extra_imgs[self.idx[i]])

def make_student():
    m = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
    m.classifier[3] = nn.Linear(m.classifier[3].in_features, 3)
    return m.to(device)

def forever(dl):
    while True:
        yield from dl

@torch.no_grad()
def recalibrate_bn(model, idx):
    # Colposcopy photos shift BatchNorm statistics; recompute them on VIA training images only
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.reset_running_stats()
            m.momentum = None
    model.train()
    for x, _ in DataLoader(ImgDS(idx, train_tf), batch_size=BATCH, num_workers=2):
        model(x.to(device))
    return model

def distill(teacher, idx, extra_idx=(), epochs=S_EPOCHS, lr=LR):
    # Teacher and student see the same augmented view; unlabelled photos only contribute the soft-label loss
    teacher.eval()
    student = make_student()
    head = [p for n, p in student.named_parameters() if n.startswith('classifier')]
    body = [p for n, p in student.named_parameters() if not n.startswith('classifier')]
    opt = torch.optim.AdamW([{'params': body, 'lr': lr * 0.3}, {'params': head, 'lr': lr}], weight_decay=1e-4)
    dl = DataLoader(ImgDS(idx, train_tf), batch_size=BATCH, shuffle=True, num_workers=2, drop_last=True)
    xs = forever(DataLoader(ExtraDS(extra_idx), batch_size=X_BATCH, shuffle=True, num_workers=2, drop_last=True)) if len(extra_idx) else None
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[lr * 0.3, lr], total_steps=epochs * len(dl), pct_start=0.15)
    counts = np.bincount(labels[idx], minlength=3)
    weights = torch.tensor(len(idx) / (3 * counts), dtype=torch.float32, device=device)
    ce_fn = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)
    scaler = torch.amp.GradScaler('cuda')
    for _ in range(epochs):
        student.train()
        for x, y in dl:
            x, y = x.to(device), torch.as_tensor(y).to(device)
            if xs is not None:
                x = torch.cat([x, next(xs).to(device)])
            with torch.autocast('cuda', dtype=torch.float16):
                with torch.no_grad():
                    t = teacher(x).float()
                s = student(x).float()
                kd = F.kl_div(F.log_softmax(s / KD_T, 1), F.softmax(t / KD_T, 1), reduction='batchmean') * KD_T ** 2
                loss = KD_ALPHA * ce_fn(s[:len(y)], y) + (1 - KD_ALPHA) * kd
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
    return recalibrate_bn(student, idx)

names = ['Teacher: EfficientNet-B3', 'Student: MobileNetV3-L, no distillation',
         'Student: + distillation (labelled VIA only)', 'Student: + distillation (all datasets)']
dprobs = {name: np.zeros((len(df), 3)) for name in names}

seed_all()
t0 = time.time()
for k in range(N_FOLDS):
    tr, te = np.where(folds != k)[0], np.where(folds == k)[0]
    test_cases = set(df['id'].iloc[te])  # other photos of a test patient never enter training
    extra_idx = np.array([i for i, c in enumerate(extra_case) if c not in test_cases])
    teacher = train(make_b3(), tr, EPOCHS, LR)
    dprobs[names[0]][te] = predict(teacher, te)
    dprobs[names[1]][te] = predict(recalibrate_bn(train(make_student(), tr, S_EPOCHS, LR), tr), te)
    dprobs[names[2]][te] = predict(distill(teacher, tr), te)
    dprobs[names[3]][te] = predict(distill(teacher, tr, extra_idx), te)
    del teacher
    torch.cuda.empty_cache()
    print(f'fold {k + 1}/{N_FOLDS}: {len(tr)} labelled + {len(extra_idx)} unlabelled training images ({(time.time() - t0) / 60:.1f} min)')

In [ ]:
# Teacher vs students: accuracy and phone size (int8 size in MB is about params in millions)
sizes = {'teacher': tp.utils.count_ops_and_params(make_b3().eval(), example),
         'student': tp.utils.count_ops_and_params(make_student().eval(), example)}
dres = pd.DataFrame({name: all_metrics(p) for name, p in dprobs.items()}).T.round(3)
macs, params = zip(*[sizes['teacher' if name.startswith('Teacher') else 'student'] for name in dres.index])
dres.insert(0, 'GMACs', np.round(np.array(macs) / 1e9, 2))
dres.insert(0, 'params (M)', np.round(np.array(params) / 1e6, 2))
dres['meets target'] = dres['refer recall (sens)'] >= MIN_SENS
dres.index.name = 'model'
os.makedirs(f'{TEAM}/results/distillation', exist_ok=True)
dres.to_csv(f'{TEAM}/results/distillation/results.csv')
dres

## Kavin - bigger models for the 4 GB phone (Nov 11)
B3 is our best model so far, but a 4 GB budget phone can run bigger ones. This checks whether we're leaving phone compute unused:

1. **Bigger single models:** EfficientNet-B4 (~2.4x B3's compute, ~1.5-4 s per photo on a budget phone), EfficientNetV2-S (~4.5x at 384 px) and EfficientNet-B5 (~5.5x, ~4-10 s). Each runs at its native input size.
2. **Team (ensemble):** the average of B3, B4, V2-S and B5. Too slow to ship, but it's the strongest teacher we can make.
3. **Distillation into B4:** one B4 trained to copy the team's predictions. B4 is the biggest model that runs comfortably on a budget phone.

- Same 300 labelled images, 5 folds, training recipe and metrics as every result above. Images are reloaded at 480 px (short side) so B5's 456 px input isn't upscaled.
- Distillation uses labelled VIA images only (the colposcopy extras hurt last time). Each teacher sees the student's augmented crop, resized to its own input size. Loss: 0.3 x cross-entropy + 0.7 x KL divergence to the team's softened predictions (temperature 2), 20 epochs.
- Out-of-fold predictions are saved to AI ML Final Project/results/bigger_models after each model, so finished models survive a disconnect. The fold checkpoints the distillation needs stay on the runtime disk.
- Decision rule: ship a bigger model only if it beats B3 by more than run-to-run noise (~0.02 AUC); otherwise B3 is 2-3x faster for the same accuracy.
- Needs cells 0, 1, 5 and 6 run first. About 2 hours for the four models and 1 hour for the distillation on a T4.

### Results (5-fold CV, 300 labelled VIA images)
The free Colab runtime ran out of GPU time during B5, so B5 and the distillation didn't finish. B3, B4 and V2-S completed and their predictions are saved in results/bigger_models.

| Model | Params | Compute vs B3 | float16 size | AUC | Sens @ 86.7% spec | Refer F1 | 3-class acc | 3-class macro F1 |
|---|---|---|---|---|---|---|---|---|
| EfficientNet-B3 | 10.7M | 1x | ~21 MB | 0.879 | 0.71 | 0.774 | 0.750 | 0.742 |
| EfficientNet-B4 | 17.6M | ~2.4x | ~35 MB | 0.852 | 0.69 | 0.764 | 0.733 | 0.727 |
| EfficientNetV2-S | 20.2M | ~4.5x | ~40 MB | 0.874 | 0.72 | 0.783 | 0.767 | 0.751 |
| **Team: B3 + V2-S (average)** | 30.9M | ~5.5x | ~62 MB | **0.899** | **0.74** | 0.801 | 0.770 | 0.767 |
| **Team: B3 + B4 + V2-S (average)** | 48.5M | ~8x | ~97 MB | **0.899** | **0.76** | **0.808** | **0.773** | **0.769** |

- **Bigger single models don't help.** B4 is worse than B3 (-0.026 AUC; bootstrap 95% CI -0.061 to +0.007) and V2-S is about the same (-0.005). With 300 labelled images, data is the limit, not model size.
- **A team of different models does help.** Averaging B3, B4 and V2-S gives +0.021 AUC over B3 (95% CI +0.001 to +0.043) and meets the 73% sensitivity target. B3 + V2-S alone gets almost all of the gain.
- **Phone cost:** B3 + V2-S is ~5.5x B3's compute (roughly 3-8 s per photo on a budget phone, ~62 MB). The two models can run one after the other, so peak memory stays under ~300 MB on a 4 GB phone.
- **Next:** distil the team into a single B3 (keeps B3's speed) once Colab GPU time is available again, and compare it with shipping the team directly.
- Thresholds are picked on the same out-of-fold predictions they're scored on, so sensitivity and F1 are slightly optimistic for every row.

In [ ]:
# Bigger models: B4, B3, EfficientNetV2-S and B5 on the same 5 folds (out-of-fold predictions saved after each model)
RES = f'{TEAM}/results/bigger_models'
CKPT = '/content/ckpt'
os.makedirs(RES, exist_ok=True)
os.makedirs(CKPT, exist_ok=True)

ZOO = {  # name: (constructor, ImageNet weights, native input size)
    'EfficientNet-B4': (models.efficientnet_b4, models.EfficientNet_B4_Weights.IMAGENET1K_V1, 380),
    'EfficientNet-B3': (models.efficientnet_b3, models.EfficientNet_B3_Weights.IMAGENET1K_V1, 300),
    'EfficientNetV2-S': (models.efficientnet_v2_s, models.EfficientNet_V2_S_Weights.IMAGENET1K_V1, 384),
    'EfficientNet-B5': (models.efficientnet_b5, models.EfficientNet_B5_Weights.IMAGENET1K_V1, 456),
}

images = [shrink(open_image(r), 480) for _, r in df.iterrows()]

def set_size(s):
    # train() and predict() read these globals
    global IMG, train_tf, eval_tf
    IMG = s
    train_tf = T.Compose([
        T.RandomResizedCrop(s, scale=(0.6, 1.0), ratio=(0.8, 1.25)),
        T.RandomHorizontalFlip(), T.RandomRotation(15),
        T.ColorJitter(0.2, 0.2, 0.1, 0.02),
        T.ToTensor(), T.Normalize(MEAN, STD)])
    eval_tf = T.Compose([T.Resize(s), T.CenterCrop(s), T.ToTensor(), T.Normalize(MEAN, STD)])

def make(name):
    fn, weights, _ = ZOO[name]
    m = fn(weights=weights)
    m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, 3)
    return m.to(device)

def fit(name, idx):
    # B5 at 456 px may not fit 16 images on a T4; fall back to batches of 8
    global BATCH
    set_size(ZOO[name][2])
    try:
        return train(make(name), idx, EPOCHS, LR)
    except torch.cuda.OutOfMemoryError:
        pass
    torch.cuda.empty_cache()
    BATCH = 8
    try:
        return train(make(name), idx, EPOCHS, LR)
    finally:
        BATCH = 16

oof = {}
t0 = time.time()
for name in ZOO:
    seed_all()
    p = np.zeros((len(df), 3))
    for k in range(N_FOLDS):
        tr, te = np.where(folds != k)[0], np.where(folds == k)[0]
        model = fit(name, tr)
        p[te] = predict(model, te)
        torch.save(model.half().state_dict(), f'{CKPT}/{name}_fold{k}.pt')
        del model
        torch.cuda.empty_cache()
        print(f'{name} fold {k + 1}/{N_FOLDS} ({(time.time() - t0) / 60:.0f} min)')
    oof[name] = p
    np.save(f'{RES}/{name}.npy', p)
    print(f'==> {name}: AUC {roc_auc_score(refer, p[:, 1] + p[:, 2]):.3f}, sens @ 86.7% spec {sens_at_spec(refer, p[:, 1] + p[:, 2], MIN_SPEC):.2f}')

In [ ]:
# Distillation: average of the four models (teacher team) -> one EfficientNet-B4 student, same folds
import torch.nn.functional as F

KD_T, KD_ALPHA, KD_EPOCHS = 2.0, 0.3, 20
STUDENT, KD_NAME = 'EfficientNet-B4', 'B4 distilled from team'

def load_teachers(k):
    out = []
    for name in ZOO:
        m = make(name)
        m.load_state_dict(torch.load(f'{CKPT}/{name}_fold{k}.pt'))
        out.append((m.eval(), ZOO[name][2]))
    return out

def distill_team(teachers, idx, epochs=KD_EPOCHS, lr=LR):
    set_size(ZOO[STUDENT][2])
    student = make(STUDENT)
    head = [p for n, p in student.named_parameters() if n.startswith('classifier')]
    body = [p for n, p in student.named_parameters() if not n.startswith('classifier')]
    opt = torch.optim.AdamW([{'params': body, 'lr': lr * 0.3}, {'params': head, 'lr': lr}], weight_decay=1e-4)
    dl = DataLoader(ImgDS(idx, train_tf), batch_size=BATCH, shuffle=True, num_workers=2, drop_last=True)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[lr * 0.3, lr], total_steps=epochs * len(dl), pct_start=0.15)
    counts = np.bincount(labels[idx], minlength=3)
    weights = torch.tensor(len(idx) / (3 * counts), dtype=torch.float32, device=device)
    ce_fn = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)
    scaler = torch.amp.GradScaler('cuda')
    for _ in range(epochs):
        student.train()
        for x, y in dl:
            x, y = x.to(device), torch.as_tensor(y).to(device)
            with torch.autocast('cuda', dtype=torch.float16):
                with torch.no_grad():
                    t = torch.stack([m(F.interpolate(x, size=(s, s), mode='bilinear', align_corners=False)).float().div(KD_T).softmax(1)
                                     for m, s in teachers]).mean(0)
                s = student(x).float()
                kd = F.kl_div(F.log_softmax(s / KD_T, 1), t, reduction='batchmean') * KD_T ** 2
                loss = KD_ALPHA * ce_fn(s, y) + (1 - KD_ALPHA) * kd
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
    return student

seed_all()
t0 = time.time()
p = np.zeros((len(df), 3))
for k in range(N_FOLDS):
    tr, te = np.where(folds != k)[0], np.where(folds == k)[0]
    teachers = load_teachers(k)
    student = distill_team(teachers, tr)
    p[te] = predict(student, te)
    del teachers, student
    torch.cuda.empty_cache()
    print(f'fold {k + 1}/{N_FOLDS} ({(time.time() - t0) / 60:.0f} min)')
oof[KD_NAME] = p
np.save(f'{RES}/{KD_NAME}.npy', p)

In [ ]:
# Bigger models vs B3: accuracy and phone cost (reads the saved predictions, so it works after a disconnect)
from sklearn.metrics import precision_recall_fscore_support

# Same metric definitions as the full-metric cell above, repeated so this section runs on its own
def op_threshold(y, s, spec):
    fpr, tpr, thr = roc_curve(y, s)
    ok = fpr <= 1 - spec + 1e-9
    return thr[ok][tpr[ok].argmax()]

def all_metrics(p):
    s = p[:, 1] + p[:, 2]
    pred, y = s >= op_threshold(refer, s, MIN_SPEC), refer == 1
    tp_, fp_, fn_, tn_ = (pred & y).sum(), (pred & ~y).sum(), (~pred & y).sum(), (~pred & ~y).sum()
    prec, rec = tp_ / max(tp_ + fp_, 1), tp_ / (tp_ + fn_)
    p3, r3, f3, _ = precision_recall_fscore_support(labels, p.argmax(1), average='macro', zero_division=0)
    return {'AUC': roc_auc_score(refer, s),
            'refer recall (sens)': rec, 'refer specificity': tn_ / (tn_ + fp_),
            'refer precision': prec, 'refer F1': 2 * prec * rec / max(prec + rec, 1e-9),
            'refer accuracy': (tp_ + tn_) / len(y),
            '3-class acc': accuracy_score(labels, p.argmax(1)),
            '3-class macro precision': p3, '3-class macro recall': r3, '3-class macro F1': f3}

TEAM_NAME = 'Team average (4 models)'
found = {n: np.load(f'{RES}/{n}.npy') for n in [*ZOO, KD_NAME] if os.path.exists(f'{RES}/{n}.npy')}
if all(n in found for n in ZOO):
    found[TEAM_NAME] = np.mean([found[n] for n in ZOO], axis=0)

cost = {}
for n in ZOO:
    s = ZOO[n][2]
    cost[n] = tp.utils.count_ops_and_params(make(n).eval(), torch.randn(1, 3, s, s, device=device))
cost[KD_NAME] = cost[STUDENT]
cost[TEAM_NAME] = tuple(np.sum([cost[n] for n in ZOO], axis=0))

bres = pd.DataFrame({n: all_metrics(p) for n, p in found.items()}).T.round(3)
macs, params = zip(*[cost[n] for n in bres.index])
bres.insert(0, 'compute vs B3', np.round(np.array(macs) / cost['EfficientNet-B3'][0], 1))
bres.insert(0, 'float16 size (MB)', np.round(np.array(params) * 2 / 1e6).astype(int))
bres.insert(0, 'params (M)', np.round(np.array(params) / 1e6, 1))
bres['meets target'] = bres['refer recall (sens)'] >= MIN_SENS
bres.index.name = 'model'
bres.to_csv(f'{RES}/results.csv')
bres